# 04-joins

04-joins/01-joins-and-broadcast.py
Join patterns: inner, left, right, full outer, semi, anti, non-equality,
self-join, broadcast, NULL key handling, column deduplication, multi-table chain.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath(__file__)), '..', '00-setup'))
from spark_session import get_spark, output_path, stop_and_wait
from seed_data import load_all, register_views
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window

spark = get_spark("04-joins")
dfs = register_views(spark)
emp  = dfs["employees"]
dept = dfs["departments"]
proj = dfs["projects"]
pa   = dfs["project_assignments"]
pr   = dfs["performance_reviews"]
po   = dfs["purchase_orders"]


# ── Section 1: INNER JOIN (employees + departments) ───────────────────────────
# Only rows with matching dept_id on both sides survive.
# emp 1 (CEO, dept_id=8 Executive) is included; emp with dept_id=None would be dropped.
print("\n=== Section 1: INNER JOIN ===")
joined = emp.join(dept, "dept_id", "inner")
joined.select("emp_id", "first_name", "dept_name", "salary").show(5)

emp_count   = emp.count()
inner_count = joined.count()
print(f"emp.count()={emp_count}  inner_join.count()={inner_count}")
# All 41 employees have a non-NULL dept_id that matches a department row, so counts match here.
# Any employee with dept_id=None (or dept_id not in departments) would be excluded.


# ── Section 2: LEFT JOIN (all employees, dept may be NULL) ────────────────────
# Disambiguate dept_id: use df.col syntax when both sides have the same column name.
# Using "on expression" style retains both emp.dept_id and dept.dept_id in result.
print("\n=== Section 2: LEFT JOIN ===")
emp.join(dept, emp.dept_id == dept.dept_id, "left") \
   .select(emp.emp_id, emp.first_name, dept.dept_name) \
   .show(5)
# If an employee had dept_id=None, dept_name would appear as NULL here.


# ── Section 3: RIGHT JOIN (all departments, including those with no employees) ─
# Departments with no employees in the data would surface with NULL emp_id/first_name.
# All 8 departments have at least one employee in this dataset.
print("\n=== Section 3: RIGHT JOIN ===")
dept.join(emp, "dept_id", "right") \
    .select("dept_name", "emp_id", "first_name") \
    .orderBy("dept_name") \
    .show(10)


# ── Section 4: FULL OUTER JOIN ────────────────────────────────────────────────
# Rows unmatched on either side appear with NULLs for the opposite table's columns.
# Alias dept.dept_id to avoid ambiguity in select when using expression-style join.
print("\n=== Section 4: FULL OUTER JOIN ===")
emp.join(dept, emp.dept_id == dept.dept_id, "full") \
   .select(
       emp.emp_id,
       emp.first_name,
       dept.dept_id.alias("dept_dept_id"),
       dept.dept_name,
   ) \
   .show(5)


# ── Section 5: LEFT ANTI-JOIN (employees with NO project assignment) ──────────
# Returns only rows from the left side that have NO match on the right.
# project_assignments covers 24 distinct employees; the rest surface here.
print("\n=== Section 5: LEFT ANTI-JOIN — employees with no project assignment ===")
assigned = pa.select("emp_id").distinct()
emp.join(assigned, "emp_id", "left_anti") \
   .select("emp_id", "first_name", "job_title") \
   .orderBy("emp_id") \
   .show()
# Most employees are not assigned to any project — expect ~17 rows.


# ── Section 6: LEFT SEMI-JOIN (employees WITH at least one assignment) ────────
# Like INNER JOIN but returns only left-side columns; no duplicate rows per match.
print("\n=== Section 6: LEFT SEMI-JOIN — employees with at least one project ===")
emp.join(assigned, "emp_id", "left_semi") \
   .select("emp_id", "first_name") \
   .orderBy("emp_id") \
   .show()


# ── Section 7: Non-equality join (employees earning more than their manager) ───
# Standard equality joins cannot express this; use a compound condition instead.
# Note: emp 10 and 15 have NULL salary — NULL comparisons evaluate to NULL (false),
# so those employees are silently excluded from the result.
print("\n=== Section 7: NON-EQUALITY JOIN — employees earning more than manager ===")
mgr = emp.select(
    F.col("emp_id").alias("mgr_id"),
    F.col("salary").alias("mgr_salary"),
)
emp.join(mgr, (emp.manager_id == mgr.mgr_id) & (emp.salary > mgr.mgr_salary), "inner") \
   .select(emp.emp_id, emp.first_name, emp.salary, mgr.mgr_id, mgr.mgr_salary) \
   .orderBy(emp.emp_id) \
   .show()


# ── Section 8: Self-join (employee + their manager name) ─────────────────────
# Re-alias the same DataFrame to create a logical "manager lookup" table.
# LEFT join so employees with no manager (e.g. CEO emp_id=1, manager_id=None)
# still appear; manager_name will be NULL for the top of the org.
print("\n=== Section 8: SELF-JOIN — employee with manager name ===")
mgr_lookup = emp.select(
    F.col("emp_id").alias("mgr_id"),
    F.col("first_name").alias("mgr_fname"),
    F.col("last_name").alias("mgr_lname"),
)
emp.join(mgr_lookup, emp.manager_id == mgr_lookup.mgr_id, "left") \
   .select(
       emp.emp_id,
       emp.first_name,
       F.concat(mgr_lookup.mgr_fname, F.lit(" "), mgr_lookup.mgr_lname).alias("manager_name"),
   ) \
   .orderBy(emp.emp_id) \
   .show(10)


# ── Section 9: Broadcast join (small dimension table sent to all workers) ──────
# Broadcast hint tells Spark to ship the small DataFrame to every executor,
# avoiding a shuffle on the large table. Verify with .explain().
print("\n=== Section 9: BROADCAST JOIN ===")
from pyspark.sql.functions import broadcast

emp.join(broadcast(dept), "dept_id", "left") \
   .select("emp_id", "first_name", "dept_name") \
   .show(5)

print("\n--- EXPLAIN (look for BroadcastHashJoin) ---")
emp.join(broadcast(dept), "dept_id", "left").explain()
# Physical plan should show: BroadcastHashJoin [...], BuildRight


# ── Section 10: NULL in join key ──────────────────────────────────────────────
# purchase_orders row 25 has dept_id=None.
# INNER JOIN: NULL != anything, so row 25 is dropped.
# LEFT JOIN (expression style): row 25 is preserved with NULL dept columns.
print("\n=== Section 10: NULL join key — purchase_orders ===")
print("PO count:", po.count())                                          # 25
print("PO INNER JOIN dept count:",
      po.join(dept, "dept_id", "inner").count())                        # 24 (row 25 dropped)
print("PO LEFT JOIN dept count:",
      po.join(dept, po.dept_id == dept.dept_id, "left").count())        # 25 (row 25 preserved)


# ── Section 11: "using" vs "on" — column deduplication ───────────────────────
# String/list style ("using"): merges the join key into a single column.
# Expression style ("on"):     keeps both sides' columns — risk of ambiguity.
print("\n=== Section 11: column deduplication — 'using' vs 'on' ===")
using_cols = emp.join(dept, ["dept_id"], "left").columns
on_cols    = emp.join(dept, emp.dept_id == dept.dept_id, "left").columns

print("'using' style columns:", using_cols)
# dept_id appears once
print("'on' style columns:", on_cols)
# emp.dept_id AND dept.dept_id both present (same name, different lineage)
# Selecting "dept_id" on the 'on' result raises AnalysisException — use df.dept_id


# ── Section 12: Multi-table join chain ────────────────────────────────────────
# Chain four tables: employees → departments → project_assignments → projects.
# All left joins so employees without assignments or projects still appear.
# project_assignments rows 25-26 are exact duplicates (emp 9, project 9) — an
# employee can appear twice here without deduplication.
print("\n=== Section 12: MULTI-TABLE JOIN CHAIN ===")
emp.join(dept, "dept_id", "left") \
   .join(pa,   "emp_id",  "left") \
   .join(proj, "project_id", "left") \
   .select(emp.emp_id, emp.first_name, dept.dept_name, proj.project_name) \
   .orderBy(emp.emp_id) \
   .show(10)
# emp 9 appears twice because of the duplicate assignment rows 25 & 26.


# ── Keep session alive for Spark UI inspection ────────────────────────────────
stop_and_wait(spark)